![tower_bridge](tower_bridge.jpeg)

As the climate changes, predicting the weather becomes ever more important for businesses. Since the weather depends on a lot of different factors, you will want to run a lot of experiments to determine what the best approach is to predict the weather. In this project, you will run experiments for different regression models predicting the mean temperature, using a combination of `sklearn` and `MLflow`.

You will be working with data stored in `london_weather.csv`, which contains the following columns:
- **date** - recorded date of measurement - (**int**)
- **cloud_cover** - cloud cover measurement in oktas - (**float**)
- **sunshine** - sunshine measurement in hours (hrs) - (**float**)
- **global_radiation** - irradiance measurement in Watt per square meter (W/m2) - (**float**)
- **max_temp** - maximum temperature recorded in degrees Celsius (°C) - (**float**)
- **mean_temp** - mean temperature in degrees Celsius (°C) - (**float**)
- **min_temp** - minimum temperature recorded in degrees Celsius (°C) - (**float**)
- **precipitation** - precipitation measurement in millimeters (mm) - (**float**)
- **pressure** - pressure measurement in Pascals (Pa) - (**float**)
- **snow_depth** - snow depth measurement in centimeters (cm) - (**float**)

In [54]:
# Run this cell to import the modules you require
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

# Read in the data
weather = pd.read_csv("london_weather.csv")

# Start coding here
# Use as many cells as you like

In [55]:
corr = weather.corr(numeric_only=True)
target_corr = corr['mean_temp'].sort_values(ascending=False)
print(target_corr)


mean_temp           1.000000
min_temp            0.955593
max_temp            0.912200
global_radiation    0.635432
sunshine            0.396535
date                0.094389
pressure            0.004764
precipitation      -0.010462
cloud_cover        -0.110556
snow_depth         -0.154945
Name: mean_temp, dtype: float64


In [56]:
weather = weather.dropna(subset=["mean_temp"])
weather.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15305 entries, 0 to 15340
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              15305 non-null  int64  
 1   cloud_cover       15286 non-null  float64
 2   sunshine          15305 non-null  float64
 3   global_radiation  15286 non-null  float64
 4   max_temp          15305 non-null  float64
 5   mean_temp         15305 non-null  float64
 6   min_temp          15305 non-null  float64
 7   precipitation     15303 non-null  float64
 8   pressure          15301 non-null  float64
 9   snow_depth        13881 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 1.3 MB


In [57]:
weather = weather.dropna(subset=["cloud_cover", "global_radiation", "precipitation", "pressure"])
weather.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 15261 entries, 0 to 15340
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              15261 non-null  int64  
 1   cloud_cover       15261 non-null  float64
 2   sunshine          15261 non-null  float64
 3   global_radiation  15261 non-null  float64
 4   max_temp          15261 non-null  float64
 5   mean_temp         15261 non-null  float64
 6   min_temp          15261 non-null  float64
 7   precipitation     15261 non-null  float64
 8   pressure          15261 non-null  float64
 9   snow_depth        13843 non-null  float64
dtypes: float64(9), int64(1)
memory usage: 1.3 MB


In [58]:
weather = weather.drop(columns=["snow_depth"])

In [59]:
print("\nMissing Values")
print(weather.isnull().sum())


Missing Values
date                0
cloud_cover         0
sunshine            0
global_radiation    0
max_temp            0
mean_temp           0
min_temp            0
precipitation       0
pressure            0
dtype: int64


In [60]:
# Feature & Target
X = weather.drop(columns=["mean_temp"])
y = weather["mean_temp"]

In [61]:
# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [62]:
#  MLFLOW Experiment Setup
mlflow.set_experiment("London_Temperature_prediction")

<Experiment: artifact_location='file:///work/files/workspace/mlruns/675463491958421853', creation_time=1778866743432, experiment_id='675463491958421853', last_update_time=1778866743432, lifecycle_stage='active', name='London_Temperature_prediction', tags={}>

In [63]:
# ==========================================
# DEFINE MODELS + HYPERPARAMETERS
# ==========================================
models = {

    "LinearRegression": {
        "model": LinearRegression(),
        "params": {}
    },

    "DecisionTree": {
        "model": DecisionTreeRegressor(
            random_state=42
        ),
        "params": {
            "max_depth": 12,
            "min_samples_split": 5
        }
    },

    "RandomForest": {
        "model": RandomForestRegressor(
            random_state=42,
            n_jobs=-1
        ),
        "params": {
            "n_estimators": 500,
            "max_depth": 25,
            "min_samples_split": 3
        }
    }
}


In [64]:
# Track Best Model
best_rmse = float("inf")
best_model = None
best_model_name = None

In [65]:
# Training Loop
for model_name, model_info in models.items():

    # Extract model and parameters
    model = model_info["model"]
    params = model_info["params"]

    # Set model hyperparameters
    if params:
        model.set_params(**params)

    # Start Mlflow run
    with mlflow.start_run(run_name=model_name):
        # build pipeline
        model_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("model", model)
        ])
        
        # Train model
        model_pipeline.fit(X_train, y_train)
        
        # Make predictions
        preds = model_pipeline.predict(X_test)
        
        # Calculate RMSE
        rmse = np.sqrt(
        mean_squared_error(y_test, preds)
        )
        
        # Log model Name
        mlflow.log_param(
        "model_name",
        model_name
        )
        # Log Hyperparameters
        for param_name, param_value in params.items():
            mlflow.log_param(
            param_name,
            param_value
            )
            
        # Log Metrics important: include 'rmse'
        mlflow.log_metric(
            "rmse",
            rmse
            )

        # Log Model
        mlflow.sklearn.log_model(
            sk_model=model_pipeline,
            artifact_path="model"
        )

        # print results
        print(
            f"{model_name} | "
            f"RMSE: {rmse:.4f}"
        )

        # Track Best Model
        if rmse < best_rmse:

            best_rmse = rmse
            best_model = model_pipeline
            best_model_name = model_name

LinearRegression | RMSE: 0.8810
DecisionTree | RMSE: 1.0811
RandomForest | RMSE: 0.8599


In [66]:
# Final Results
print("\n===================================")
print(f"Best Model : {best_model_name}")
print(f"Best RMSE  : {best_rmse:.4f}")
print("===================================")



Best Model : RandomForest
Best RMSE  : 0.8599


In [67]:
# Target Check
if best_rmse <= 3:
    print("SUCCESS: RMSE <= 3 achieved")

else:
    print(
        "Target NOT achieved yet.\n"
        "Try additional feature engineering "
        "or hyperparameter tuning."
    )



SUCCESS: RMSE <= 3 achieved


In [68]:
# Search al Mlflow runs
experiment = mlflow.get_experiment_by_name(
    "London_Temperature_prediction"
)

experiment_results = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

# Display results
print("\nMLflow Experiment Results:")

print(
    experiment_results[
        [
            "run_id",
            "params.model_name",
            "metrics.rmse"
        ]
    ]
)


MLflow Experiment Results:
                             run_id params.model_name  metrics.rmse
0  9745f7771e504075b8472fa75fae16f7      RandomForest      0.859942
1  31af124a8a75418fb3d02ebf56895847      DecisionTree      1.081086
2  0d9e4b04471949ef8bfaf24c6f5233fc  LinearRegression      0.880979
3  ac8bf3efdd6d4eefbe8218e2110aa742      RandomForest      0.859942
4  cbf0566004154982aba970e31a2e166e      DecisionTree      1.081086
5  36e23d26476343d69f0f6f52bdbc0ae9  LinearRegression      0.880979
6  6854698f2d7d47959674179c176121ae              None           NaN
7  4d09221c5e424c7f930dd2917969971d              None           NaN
